In [ ]:
!pip install -U "transformers>=4.44" "datasets>=2.19" "accelerate>=0.34" peft bitsandbytes trl


In [ ]:
# 0) ambiente y seeds
import os, random, math, torch, pandas as pd
SEED = 123
random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

PyTorch: 2.8.0+cu126 | device: cuda


In [ ]:
# ============================================================
# 1) Carga tokenizer + modelo Phi-3.5-mini-instruct en 4-bit
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Modelo base
MODEL_ID = "microsoft/phi-3.5-mini-instruct"

# Verificación para usar bf16 si la GPU lo soporta (Compute Capability >= 8, ej. Ada / Ampere)
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

# Configuración de cuantización (igual que Qwen3)
quant_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16_ok else torch.float16,
)

print("[1] Cargando tokenizer…")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    trust_remote_code=True
)

# Carga el modelo en 4-bit con asignación automática a GPU/CPU según disponibilidad
print("[1] Cargando modelo (4-bit)…")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=quant_4bit,
    trust_remote_code=True
)

# Algunos modelos de Phi no traen pad_token definido, lo igualamos al eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# El modelo Phi-3.5-mini tiene un contexto máximo de 16K
MODEL_MAX = getattr(model.config, "max_position_embeddings", 16384)
tokenizer.model_max_length = MODEL_MAX

# Para entrenamiento ligero, reducir longitud de secuencia
MAX_SEQ_LEN_TRAIN = 2048
print("[1] MODEL_MAX:", MODEL_MAX, "| MAX_SEQ_LEN_TRAIN:", MAX_SEQ_LEN_TRAIN)


[1] Cargando tokenizer…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[1] Cargando modelo (4-bit)…


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[1] MODEL_MAX: 131072 | MAX_SEQ_LEN_TRAIN: 2048


In [ ]:
# ============================================================
# 2) Preparar k-bit + gradient checkpointing + grads en input
# ============================================================

from peft import prepare_model_for_kbit_training

print("[2] Preparando modelo para k-bit training…")

# Ajusta la preparación para entrenamiento en 4-bit (PEFT)
# Esto coloca los pesos en modo no entrenable, prepara norm layers y dtype correcto
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Activar gradient checkpointing para reducir memoria VRAM
model.gradient_checkpointing_enable()

# Desactivar la caché de atención (necesario para entrenamiento)
model.config.use_cache = False

# Algunos modelos requieren habilitar gradientes en las entradas
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    def make_inputs_require_grad(module, input, output):
        output.requires_grad_(True)
    model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

print("[2] Listo para fine-tuning en 4-bit ✅")


[2] Preparando modelo para k-bit training…
[2] Listo para fine-tuning en 4-bit ✅


In [ ]:
# ============================================================
# 3) Detectar módulos de atención y aplicar LoRA (Phi-3.5)
# ============================================================

from peft import LoraConfig, get_peft_model

# Módulos más comunes en la arquitectura Phi-3.5 (Tiny-Transformer optimizado)
# Basado en inspección de model.named_modules()
default_targets = [
    "q_proj", "k_proj", "v_proj", "o_proj",           # comunes en modelos LLaMA/Qwen
    "Wqkv", "out_proj",                              # usados en Phi 2/3
    "linear", "mlp.fc1", "mlp.fc2"                   # fallback (algunas versiones)
]

# Autodetección
all_module_names = [n for n, _ in model.named_modules()]
selected = [t for t in default_targets if any(n.endswith(t) for n in all_module_names)]

# Fallback adicional para Phi (casos donde hay 'mixer.Wqkv' y 'mixer.out_proj')
if not selected:
    phi_targets = [n for n in all_module_names if any(k in n for k in ["Wqkv", "out_proj"])]
    if phi_targets:
        selected = ["Wqkv", "out_proj"]

if not selected:
    raise ValueError("[3] No encontré módulos de atención típicos. Ejemplo de nombres:",
                     all_module_names[:40])

print("[3] target_modules LoRA:", selected)

# Configuración LoRA optimizada para Phi-3.5-mini (4-bit)
lora_cfg = LoraConfig(
    r=8,                    # rank: equilibrio entre capacidad y memoria
    lora_alpha=16,          # escala (16–32 funciona bien)
    lora_dropout=0.05,      # regularización ligera
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=selected
)

# Inyectar adaptadores LoRA
model = get_peft_model(model, lora_cfg)

# Verificar porcentaje de parámetros entrenables
def count_trainable_params(m):
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in m.parameters())
    return tr, tot, 100 * tr / tot

t, T, pct = count_trainable_params(model)
print(f"[3] Parámetros entrenables: {t:,} / {T:,} ({pct:.4f}%)")


[3] target_modules LoRA: ['v_proj', 'o_proj']
[3] Parámetros entrenables: 1,572,864 / 2,010,713,088 (0.0782%)


In [ ]:
# ============================================================
# 4) Cargar datos (CSV) y vistazo rápido
# ============================================================

import pandas as pd
from pathlib import Path

# Carpeta base con los CSV preprocesados
DATA_DIR = Path("/content/drive/MyDrive/MAIA-PROYECTO")

# Archivos individuales
train_path = DATA_DIR / "data_finetuning_train.csv"
val_path   = DATA_DIR / "data_finetuning_val.csv"
test_path  = DATA_DIR / "data_finetuning_test.csv"

# Leer CSV
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

# Información general
print(f"Train set: {len(train_df):,} muestras")
print(f"Validation set: {len(val_df):,} muestras")
print(f"Test set: {len(test_df):,} muestras")

print("Columnas disponibles:", train_df.columns.tolist())

# Vista rápida de las primeras filas
display(train_df.head(2))


Train set: 3,038 muestras
Validation set: 380 muestras
Test set: 380 muestras
Columnas disponibles: ['name', 'article', 'summary']


,name,article,summary
0,10.1002-14651858.CD010557.pub2,Background\nAlthough antidepressants are often...,Are there effective medications for treating d...
1,10.1002-14651858.CD000938.pub2,Background\nWomen with a suspected large‐for‐d...,Induction of labour at or near the end of preg...


In [ ]:
# ============================================================
# 5) Tokenización — versión optimizada para T4 en Colab
# ============================================================

from datasets import Dataset
import pandas as pd

# Configuraciones clave
USE_CHUNKING = True       # True → divide artículos largos (recomendado)
MAX_TARGET_TOKENS = 384   # resumen máximo
OVERLAP = 64              # menor solapamiento para ahorrar VRAM
MAX_SEQ_LEN_TRAIN = 2048  # ya definido antes

# Prompt base (mantén el mismo estilo del fine-tuning previo)
SYS_PROMPT = (
    "You are a helpful medical/health writer who rewrites complex biomedical text "
    "into Plain Language Summaries understandable for laypeople. "
    "Use short sentences, everyday words, and neutral tone. "
    "Avoid jargon; when unavoidable, define it simply."
)

# ------------------------------------------------------------
# CHUNKING (recomendado para textos >2 000 tokens)
# ------------------------------------------------------------
if USE_CHUNKING:
    PROMPT_PREFIX = SYS_PROMPT + "\n\nScientific Text:\n"
    PROMPT_SUFFIX = "\n\nPlain Language Summary:"

    def tokenize_chunks(article: str, summary: str):
        # tokens del target (resumen)
        tgt_ids = tokenizer(
            summary + tokenizer.eos_token,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_TARGET_TOKENS
        ).input_ids

        prefix_ids = tokenizer(PROMPT_PREFIX, add_special_tokens=False).input_ids
        suffix_ids = tokenizer(PROMPT_SUFFIX, add_special_tokens=False).input_ids

        avail_for_chunk = MAX_SEQ_LEN_TRAIN - len(tgt_ids) - len(prefix_ids) - len(suffix_ids)
        if avail_for_chunk <= 0:
            ids = tgt_ids[-MAX_SEQ_LEN_TRAIN:]
            return [{"input_ids": ids, "labels": ids[:]}]

        art_ids = tokenizer(article, add_special_tokens=False).input_ids
        stride = max(1, avail_for_chunk - OVERLAP)
        examples = []

        for start in range(0, len(art_ids), stride):
            chunk = art_ids[start:start + avail_for_chunk]
            if not chunk:
                break
            inp_ids = prefix_ids + chunk + suffix_ids
            ids = inp_ids + tgt_ids
            labels = [-100] * len(inp_ids) + tgt_ids
            examples.append({"input_ids": ids, "labels": labels})
        return examples

    def df_to_dataset(frame: pd.DataFrame):
        input_ids_list, labels_list = [], []
        for a, s in zip(frame["article"].tolist(), frame["summary"].tolist()):
            for ex in tokenize_chunks(a, s):
                input_ids_list.append(ex["input_ids"])
                labels_list.append(ex["labels"])
        ds = Dataset.from_dict({"input_ids": input_ids_list, "labels": labels_list})
        ds.set_format(type="torch", columns=["input_ids", "labels"])
        return ds

# ------------------------------------------------------------
# SIN CHUNK (solo truncado)
# ------------------------------------------------------------
else:
    def build_prompt(article: str) -> str:
        return f"{SYS_PROMPT}\n\nScientific Text:\n{article}\n\nPlain Language Summary:"

    def tokenize_one(article: str, summary: str):
        tgt = tokenizer(
            summary + tokenizer.eos_token,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_TARGET_TOKENS
        ).input_ids
        avail = MAX_SEQ_LEN_TRAIN - len(tgt)
        if avail <= 0:
            ids = tgt[-MAX_SEQ_LEN_TRAIN:]
            return {"input_ids": ids, "labels": ids[:]}
        inp = tokenizer(
            build_prompt(article),
            add_special_tokens=False,
            truncation=True,
            max_length=avail
        ).input_ids
        ids = inp + tgt
        labels = [-100] * len(inp) + tgt
        return {"input_ids": ids, "labels": labels}

    def df_to_dataset(frame: pd.DataFrame):
        input_ids_list, labels_list = [], []
        for a, s in zip(frame["article"].tolist(), frame["summary"].tolist()):
            ex = tokenize_one(a, s)
            input_ids_list.append(ex["input_ids"])
            labels_list.append(ex["labels"])
        ds = Dataset.from_dict({"input_ids": input_ids_list, "labels": labels_list})
        ds.set_format(type="torch", columns=["input_ids", "labels"])
        return ds

print(f"[5] Tokenizador listo. USE_CHUNKING = {USE_CHUNKING}")


[5] Tokenizador listo. USE_CHUNKING = True


In [ ]:
# ============================================================
# 6) Split y construcción de datasets tokenizados
# ============================================================

from datasets import DatasetDict

def safe_df_to_dataset(frame):
    """Convierte un DataFrame en Dataset HF, ignorando filas vacías."""
    frame = frame.dropna(subset=["article", "summary"])
    ds = df_to_dataset(frame)
    ds.set_format(type="torch", columns=["input_ids", "labels"])
    return ds

print("[6] Tokenizando train…")
tok_train = safe_df_to_dataset(train_df)

print("[6] Tokenizando validation…")
tok_val = safe_df_to_dataset(val_df)

print("[6] Tokenizando test…")
tok_test = safe_df_to_dataset(test_df)

# DatasetDict para usar con Trainer
tok = DatasetDict({
    "train": tok_train,
    "validation": tok_val,
    "test": tok_test
})

print(tok)
print(f"[6] Tokens por split: train={len(tok_train)}, val={len(tok_val)}, test={len(tok_test)}")
print("[6] Ejemplo:", tok_train[0])


[6] Tokenizando train…
[6] Tokenizando validation…
[6] Tokenizando test…
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 3914
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 486
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 489
    })
})
[6] Tokens por split: train=3914, val=486, test=489
[6] Ejemplo: {'input_ids': tensor([  887,   526,   263,  ...,   573, 25828,  4835]), 'labels': tensor([ -100,  -100,  -100,  ...,   573, 25828,  4835])}


In [ ]:
# ============================================================
# 7) collator + sanity forward/backward (Phi-3.5-mini, T4)
# ============================================================
from dataclasses import dataclass
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
import torch
from itertools import islice

@dataclass
class CausalCollator:
    pad_token_id: int
    def __call__(self, batch):
        ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
        ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
        am = [torch.ones_like(x) for x in ii]
        ii = pad_sequence(ii, batch_first=True, padding_value=self.pad_token_id)
        ll = pad_sequence(ll, batch_first=True, padding_value=-100)
        am = pad_sequence(am, batch_first=True, padding_value=0)
        return {"input_ids": ii, "labels": ll, "attention_mask": am}

collator = CausalCollator(pad_token_id=tokenizer.pad_token_id)
dl = DataLoader(
    tok["train"],
    batch_size=1,          # seguro para T4
    shuffle=True,
    collate_fn=collator,
    num_workers=2,         # evita problemas en Colab
    pin_memory=True
)

batch = next(iter(dl))
for k,v in batch.items():
    print("[7]", k, v.shape, v.dtype)

model.train()
scal_dtype = torch.bfloat16 if bf16_ok else torch.float16

# ------- sanity FORWARD -------
with torch.autocast(device_type="cuda", dtype=scal_dtype):
    out = model(**{k: v.to(model.device, non_blocking=True) for k,v in batch.items()})
print


/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is

[7] input_ids torch.Size([1, 2048]) torch.int64
[7] labels torch.Size([1, 2048]) torch.int64
[7] attention_mask torch.Size([1, 2048]) torch.int64


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [ ]:
# ============================================================
# 8) sanity backward (un paso manual)
# ============================================================
import torch

# Optimizador solo sobre los parámetros entrenables (LoRA)
optim = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4
)

# Tomamos un batch aleatorio
batch = next(iter(dl))
batch = {k: v.to(model.device, non_blocking=True) for k, v in batch.items()}

optim.zero_grad(set_to_none=True)

# Forward + backward en autocast (FP16 para T4)
dtype = torch.bfloat16 if bf16_ok else torch.float16
with torch.autocast(device_type="cuda", dtype=dtype):
    out = model(**batch)
loss = out.loss
print(f"[8] loss: {float(loss):.4f} | requires_grad: {loss.requires_grad}")

# Backward + step
loss.backward()
optim.step()

print("[8] backward y step OK ✅")

# (Opcional) verificar gradientes en LoRA
grad_tensors = [n for n,p in model.named_parameters() if p.requires_grad and p.grad is not None]
print(f"[8] Gradientes activos en {len(grad_tensors)} módulos LoRA (OK si >0)")


/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is

[8] loss: 1.3283 | requires_grad: True
[8] backward y step OK ✅
[8] Gradientes activos en 64 módulos LoRA (OK si >0)


In [ ]:
# ============================================================
# 9) Trainer — validación del pipeline (1 época)
# ============================================================
from transformers import Trainer, TrainingArguments
import math, torch

EPOCHS = 1
BATCH_SIZE = 1
ACCUM_STEPS = 16
WARMUP_RATIO = 0.1
SEED = 42

n_train = len(tok["train"])
steps_per_epoch = max(1, math.ceil(n_train / (BATCH_SIZE * ACCUM_STEPS)))
warmup_steps = int(WARMUP_RATIO * steps_per_epoch * EPOCHS)

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/MAIA-PROYECTO/outputs/phi3.5-mini-qlora",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCUM_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    logging_steps=30,
    do_eval=True,
    eval_steps=30,
    save_steps=30,
    save_total_limit=3,
    bf16=bf16_ok,                  # False en T4 → usa FP16 automáticamente
    fp16=not bf16_ok,
    gradient_checkpointing=True,   # mantiene memoria baja
    dataloader_pin_memory=True,
    dataloader_num_workers=0,      # evita conflictos en Colab
    seed=SEED, data_seed=SEED,
    report_to=None,                # sin W&B ni TB
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok["train"],
    eval_dataset=tok["validation"],
    data_collator=collator,
)

print(f"[9] Entrenando… pasos/época = {steps_per_epoch}")
trainer.train()
print("[9] Fin de trainer.train() ✅")


[9] Entrenando… pasos/época = 245


wandb: Currently logged in as: jsoa9012 (jsoa9012-universidad-de-los-andes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]


Step,Training Loss
30,1.387700
60,1.250600
90,1.235200
120,1.243000
150,1.220200
180,1.215900
210,1.221500
240,1.212600


/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is

[9] Fin de trainer.train() ✅


In [ ]:
#  10) Guardar modelo
# Guarda los pesos finales del modelo ajustado
trainer.save_model("/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora")

# Guarda el tokenizer para inferencia
tokenizer.save_pretrained("/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora")

('/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/tokenizer_config.json',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/special_tokens_map.json',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/chat_template.jinja',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/tokenizer.model',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/added_tokens.json',
 '/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora/tokenizer.json')

In [ ]:
# # 10) Evaluar modelo
trainer.evaluate()

/tmp/ipython-input-1335483151.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ii = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
/tmp/ipython-input-1335483151.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ll = [torch.tensor(b["labels"],    dtype=torch.long) for b in batch]


{'eval_loss': 1.1961791515350342,
 'eval_runtime': 580.344,
 'eval_samples_per_second': 0.837,
 'eval_steps_per_second': 0.837,
 'epoch': 1.0}

In [ ]:
#!pip uninstall -y bitsandbytes triton
#!pip install bitsandbytes==0.43.2 triton==2.3.0
#!pip install transformers==4.43.3 peft==0.10.0 accelerate==0.33.0
!pip install -U peft==0.13.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 21.6 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.10.0
    Uninstalling peft-0.10.0:
      Successfully uninstalled peft-0.10.0


In [2]:
!pip install -U \
  torch==2.8.0 \
  torchvision==0.23.0 \
  torchaudio==2.8.0 \
  transformers==4.57.1 \
  peft==0.17.1 \
  bitsandbytes==0.48.2 \
  accelerate==1.11.0 \
  trl==0.24.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.9 MB/s eta 0:00:00


In [2]:
# ============================================================
# Generar PLS con Phi-3.5-mini-instruct (QLoRA)
# Columnas esperadas: name, article, summary
# ============================================================

import re
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig

# ---- 1) Rutas y datos ----
DATA_DIR = Path("/content/drive/MyDrive/MAIA-PROYECTO")
RESULTS_DIR = Path("/content/drive/MyDrive/MAIA-PROYECTO/models/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_DIR / "data_finetuning_test.csv")
for col in ["name", "article", "summary"]:
    assert col in df.columns, f"Falta la columna requerida: {col}"

# ---- 2) Prompt coherente con tu entrenamiento ----
def generar_prompt(texto_cientifico):
    return f"""You are a helpful medical/health writer.
Rewrite the following scientific text into a clear, plain-language summary for a general audience.
Avoid headings, bullet points, or lists. Use short sentences, simple words, and neutral tone.
Define complex terms when necessary.

Scientific text: {texto_cientifico}

Plain Language Summary:"""

# ---- 3) Modelo y adaptador ----
BASE_MODEL = "microsoft/phi-3.5-mini-instruct"  # Tokenizer y pesos base originales
LORA_DIR   = "/content/drive/MyDrive/MAIA-PROYECTO/models/phi35-qlora"
CKPT_DIR   = "/content/drive/MyDrive/MAIA-PROYECTO/outputs/phi3.5-mini-qlora/checkpoint-240"

bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16_ok else torch.float16
)

print("[Modelo] Cargando tokenizer base…")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[Modelo] Cargando modelo base en 4-bit…")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_cfg,
    torch_dtype=torch.bfloat16 if bf16_ok else torch.float16,
    trust_remote_code=True
)

print("[Modelo] Montando adaptador LoRA fine-tuneado…")

# Explicitly load LoraConfig and remove 'corda_config' if it exists
try:
    lora_config = LoraConfig.from_pretrained(LORA_DIR)
    if hasattr(lora_config, 'corda_config'):
        delattr(lora_config, 'corda_config')
except Exception as e:
    print(f"Error loading LoRA config or removing corda_config: {e}")
    lora_config = None # Fallback if loading config fails

if lora_config:
    model = PeftModel.from_pretrained(base_model, LORA_DIR, config=lora_config)
else:
    # Attempt to load without explicit config if loading failed
    print("Attempting to load PeftModel without explicit config due to error.")
    model = PeftModel.from_pretrained(base_model, LORA_DIR)


if CKPT_DIR:
    try:
        model.load_adapter(CKPT_DIR, adapter_name="resume")
        model.set_adapter("resume")
    except Exception:
        print("⚠️ No se encontró checkpoint adicional, usando adaptador principal.")
model.eval()
print("✅ Modelo y tokenizer listos para inferencia")

# ---- 4) Parámetros de generación ----
BATCH_SIZE = 2   # puedes subirlo a 4–6 si usas tu RTX 4050
GEN_KW = dict(
    max_new_tokens=512,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    use_cache=False,             # ⚠️ evita el bug DynamicCache.seen_tokens
    return_dict_in_generate=True
)

# ---- 5) Generador por lotes ----
def generate_batch(prompts):
    enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
    input_ids = enc["input_ids"].to(model.device)
    attn_mask = enc["attention_mask"].to(model.device)
    input_lengths = attn_mask.sum(dim=1)

    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            **GEN_KW
        )

    sequences = out.sequences
    textos = []
    for i in range(sequences.size(0)):
        gen_tokens = sequences[i, input_lengths[i]:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        textos.append(texto)
    return textos



The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

[Modelo] Cargando tokenizer base…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[Modelo] Cargando modelo base en 4-bit…


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[Modelo] Montando adaptador LoRA fine-tuneado…
⚠️ No se encontró checkpoint adicional, usando adaptador principal.
✅ Modelo y tokenizer listos para inferencia


In [4]:
# ---- 6) Inferencia por lotes ----
articulos = df["article"].fillna("").astype(str).tolist()
pls = []

print(f"[Inferencia] Generando {len(articulos)} resúmenes…")
for i in tqdm(range(0, len(articulos), BATCH_SIZE)):
    batch = articulos[i:i+BATCH_SIZE]
    prompts = [generar_prompt(t) for t in batch]
    pls.extend(generate_batch(prompts))

# ---- 7) Guardar resultados ----
df["pls_phi35"] = pls
csv_out = RESULTS_DIR / "summaries_phi35.csv"
df.to_csv(csv_out, index=False, encoding="utf-8")
print(f"✅ Guardado CSV con resúmenes en: {csv_out}")

[Inferencia] Generando 380 resúmenes…


  0%|          | 0/190 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


✅ Guardado CSV con resúmenes en: /content/drive/MyDrive/MAIA-PROYECTO/models/results/summaries_phi35.csv
